In [ ]:
# Abira Demello - Statistical Analysis

In [ ]:
import pandas as pd 
import polars as pl

# MSTL - Multi Seasonal Trend decompostion using LOESS
from statsmodels.tsa.seasonal import MSTL
import warnings

In [ ]:
from tqdm import tqdm # shows a progress bar when looping through data; Useful when processing many time series
tqdm.pandas()

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [4]:
import httpx

In [ ]:
# Data Fetching - connecting to the backend API and pulling in the employment data
base_uri = 'http://127.0.0.1:8000/api/'

api_routes = {
    1: 'program_university_kpi_latest',
    2: 'program_category',
    3: 'program_noc_link',
    4: 'provincial_noc_group_labor_statistic',
    5: 'noc_grouping',
    6: 'economic_region_employment_estimate',
    7: 'economicregion',
    8: 'province',
    9: 'university',
    10: 'prediction_cache'
}

# Instantiating a synchronous client with 120s timeout
client = httpx.Client(timeout=500.0)

# Generic function for making a request to the API given a formatted URL
def Make_Request(url: str):
    try:
        print(f"URL => {url}")
        response = client.get(url)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Request failed for {url}: {e}")
        return []
# Specific data fetchers built on top of Make_Request

# Pulls the employment statistics based on NOC Group
def RequestNOCGroupLaborStatistics():
    try:
        url_to_call = f"{base_uri}{api_routes[4]}/"
        data = Make_Request(url=url_to_call)
        return pd.DataFrame(data)
    except Exception as e:
        print(f"RequestNOCGroupLaborStatistics() => {e}")
        return pd.DataFrame()

# Pulls the province reference data
def RequestProvince() -> pd.DataFrame:
    try:
        url_to_call = f"{base_uri}{api_routes[8]}/"
        data = Make_Request(url=url_to_call)
        return pd.DataFrame(data) # fetches province data and converts it into a usable table
    except Exception as e:
        print(f"RequestProvince() => {e}")
        return pd.DataFrame()

#region Request Functions

In [6]:
ProvinceData = RequestProvince()

URL => http://127.0.0.1:8000/api/province/


In [7]:
ProvinceData

,provinceid,province_dguid,province,province_shorthand
0,59,2021A000259,British Columbia,BC
1,24,2021A000224,Quebec,QC
2,11,2021A000211,Prince Edward Island,PE
3,47,2021A000247,Saskatchewan,SK
4,46,2021A000246,Manitoba,MB
5,13,2021A000213,New Brunswick,NB
6,10,2021A000210,Newfoundland and Labrador,NL
7,60,2021A000260,Yukon,YT
8,61,2021A000261,Northwest Territories,NT
9,48,2021A000248,Alberta,AB


In [ ]:
# Creating a Mapping; converts the dguid to provinceid. will map IDs to provinces when analyzing time series
ProvinceDGUID_to_ID_Map = ProvinceData.set_index('province_dguid')['provinceid'].to_dict()
ProvinceDGUID_to_ID_Map

{'2021A000259': 59,
 '2021A000224': 24,
 '2021A000211': 11,
 '2021A000247': 47,
 '2021A000246': 46,
 '2021A000213': 13,
 '2021A000210': 10,
 '2021A000260': 60,
 '2021A000261': 61,
 '2021A000248': 48,
 '2021A000212': 12,
 '2021A000262': 62,
 '2021A000235': 35}

In [ ]:
# Loading the employment data; table shows monthly employment data going back to January 2000 organized by NOC group and region (DGUID)
# Each row is one month's snapshot for one job category in one region
Data = RequestNOCGroupLaborStatistics()
Data.head()

URL => http://127.0.0.1:8000/api/provincial_noc_group_labor_statistic/


,entryid,datestamp,noc_groupingid,noc_grouping,dguid,labour_force,employment,part_time_employment,full_time_employment,unemployment,unemployment_rate,employment_moving_average,employment_moving_sd,employment_upper_band,employment_lower_band
0,0,2000-01-01 00:00:00,1,Professional occupations in health [31],2021A000011124,401.5,399.8,103.1,296.7,1.7,0.4,399.80,NaN,NaN,NaN
1,1,2000-02-01 00:00:00,1,Professional occupations in health [31],2021A000011124,404.6,402.3,97.5,304.9,2.3,0.6,401.05,1.768,404.586,397.514
2,2,2000-03-01 00:00:00,1,Professional occupations in health [31],2021A000011124,389.2,387.1,81.2,305.9,2.1,0.5,396.40,8.150,412.700,380.100
3,3,2000-04-01 00:00:00,1,Professional occupations in health [31],2021A000011124,395.3,393.6,89.7,303.9,1.7,0.4,395.70,6.800,409.300,382.100
4,4,2000-05-01 00:00:00,1,Professional occupations in health [31],2021A000011124,404.0,401.6,90.3,311.3,2.4,0.6,396.15,7.209,410.568,381.732


In [ ]:
# converts each dguid code into the internal numeric province id
# dropna drops any rows where the dguid did not match a known province

Data['provinceid'] = Data['dguid'].map(ProvinceDGUID_to_ID_Map)
Data = Data.dropna(subset=['provinceid'])

In [ ]:
# Slimming down the dataset

DataReduced = Data[['noc_groupingid','provinceid','datestamp','employment']].dropna()
DataReduced.head(10)

,noc_groupingid,provinceid,datestamp,employment
309,1,10.0,2000-01-01 00:00:00,7.0
310,1,10.0,2000-02-01 00:00:00,8.2
311,1,10.0,2000-03-01 00:00:00,7.1
312,1,10.0,2000-04-01 00:00:00,7.6
313,1,10.0,2000-05-01 00:00:00,6.9
314,1,10.0,2000-06-01 00:00:00,7.8
315,1,10.0,2000-07-01 00:00:00,7.3
316,1,10.0,2000-08-01 00:00:00,7.0
317,1,10.0,2000-09-01 00:00:00,7.9
318,1,10.0,2000-10-01 00:00:00,8.4


In [ ]:
# Creating keys: creates a unique identifier for each time series; a "key" is just a pair like (10,1) meaning province 10 and, noc group 1
# each unique combination of province + job category becomes its own seperate time series that will get decomposed independently

DataReduced['Key'] = DataReduced.progress_apply(lambda x: (x['provinceid'], x['noc_groupingid']), axis=1)

100%|██████████| 181386/181386 [00:00<00:00, 253818.95it/s]


In [ ]:
# Collects all the unique keys into a list — this becomes the list of series to loop over when running MSTL.
Keys = DataReduced['Key'].unique().tolist()

In [ ]:
# Converting the pandas DF into a Polars DF 
# Polars is significantly faster at filtering and slicing large datasets
df = pl.from_pandas(DataReduced)

In [15]:
del DataReduced

Seasonal Decomposition 

In [ ]:
#Sample: "Professional occupations in engineering [213]" in Ontario 

Sample = df.filter((pl.col('provinceid')==59) & (pl.col('noc_groupingid')==30)).select(['datestamp','employment']).sort(by=['datestamp'], descending=False).to_pandas()
Sample

,datestamp,employment
0,2000-01-01 00:00:00,20.0
1,2000-02-01 00:00:00,18.2
2,2000-03-01 00:00:00,17.8
3,2000-04-01 00:00:00,16.7
4,2000-05-01 00:00:00,17.7
...,...,...
304,2025-05-01 00:00:00,40.1
305,2025-06-01 00:00:00,38.7
306,2025-07-01 00:00:00,40.8
307,2025-08-01 00:00:00,38.3


In [ ]:
# Converts the date column into a proper datetime format so pandas and MSTL understand it as time, not just text.
Sample['datestamp'] = pd.to_datetime(Sample['datestamp'])

In [55]:
Sample.dtypes

datestamp     datetime64[ns]
employment           float64
dtype: object

In [ ]:
# this reshapes the data into a proper time series format - a single column of employment numbers where the row labels are the dates (YYYY-MM-DD) 
# MSTL requires this format to understand the time ordering

SampleTS = Sample.sort_values('datestamp')['employment']
SampleTS.index = Sample['datestamp']
SampleTS

datestamp
2000-01-01    20.0
2000-02-01    18.2
2000-03-01    17.8
2000-04-01    16.7
2000-05-01    17.7
              ... 
2025-05-01    40.1
2025-06-01    38.7
2025-07-01    40.8
2025-08-01    38.3
2025-09-01    33.9
Name: employment, Length: 309, dtype: float64

In [ ]:
# This is the decomposition. we're tellig MSTL to break this time series into components by looking for 3 different seasonal cycle lengths
# 12 - annual seasonality 
# 6 - semi annual 
# 3 - quarterly (patterns repeating every 3 months)
mstl = MSTL(SampleTS, periods=[12, 6, 3])
result = mstl.fit()

In [ ]:
# MSTL splits the original series into 4 pieces
# observed: the original raw employment numbers
observed = result.observed

# trend: the smooth long-term direction from 2000 to 2025
trend = result.trend

# the 3 seasonal patterns (3,6,12 months) - the repeating wiggles at each cycle length
seasonalities = result.seasonal

# residual : the leftover noise that none of the above couple explain
resid = result.resid

In [59]:
df_decomp = (
    observed.rename("observed")
    .to_frame()
    .join(trend.rename("trend"))
    .join(resid.rename("resid"))
    .join(seasonalities)  # already a DataFrame
)

In [ ]:
# Copy the datetime index into its own column 'ds' for explicit date access during downstream processing
df_decomp['ds'] = df_decomp.index.values
df_decomp


,observed,trend,resid,seasonal_3,seasonal_6,seasonal_12,ds
datestamp,,,,,,,
2000-01-01,20.0,18.555255,1.282449,-0.642285,0.484367,0.320215,2000-01-01
2000-02-01,18.2,18.337094,0.183746,0.066125,-0.177066,-0.209900,2000-02-01
2000-03-01,17.8,18.133526,0.756423,0.506046,-0.614850,-0.981144,2000-03-01
2000-04-01,16.7,17.945904,-0.539334,-0.528631,-0.356752,0.178813,2000-04-01
2000-05-01,17.7,17.775230,0.605678,0.048941,0.105680,-0.835529,2000-05-01
...,...,...,...,...,...,...,...
2025-05-01,40.1,37.617595,0.359809,-0.844608,1.796586,1.170618,2025-05-01
2025-06-01,38.7,38.235024,1.703189,-1.082396,-0.447340,0.291523,2025-06-01
2025-07-01,40.8,38.900544,2.834867,1.992600,0.514062,-3.442073,2025-07-01


In [ ]:
df_decomp = df_decomp.reset_index(drop=True)

In [67]:
df_decomp.dtypes

observed              float64
trend                 float64
resid                 float64
seasonal_3            float64
seasonal_6            float64
seasonal_12           float64
ds             datetime64[ns]
dtype: object

In [ ]:
# creates a 6-panel stacked chart, all sharing the time axis on the x-axis
# each panel gets one component of the decomposition 

fig = make_subplots(
    rows=6,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=[
        "Employment in Thousands",
        "Trend",
        "Annual Seasonality",
        "Semi-Annual Seasonality",
        "Quarterly Seasonality",
        "Residuals"
        
    ]
)

# Observed: solid line; raw data - shown as-is
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['observed'],
        mode="lines",
        name="Employment in Thousands",
        line=dict(color='#13256B', width=2.5)
    ),
    row=1,
    col=1
)

# Trend: dashed line; same color as observed but dashed to signal its a smoothed version
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['trend'],
        mode="lines",
        name="Trend",
        line=dict(color='#13256B', width=2.5, dash='dash')
    ),
    row=2,
    col=1
)

# Seasonal_12: solid line ; annual cycle
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['seasonal_12'],
        mode="lines",
        name="Annual Seasonality",
        line=dict(color='#064241', width=2.5)
    ),
    row=3,
    col=1
)

# Seasonal_6: semi annual
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['seasonal_6'],
        mode="lines",
        name="Semi-Annual Seasonality",
        line=dict(color='#17615E', width=2.5)
    ),
    row=4,
    col=1
)

#Seasonal_3: quarterly cycle
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['seasonal_3'],
        mode="lines",
        name="Quarterly Seasonality",
        line=dict(color="#42B3AF", width=2.5)
    ),
    row=5,
    col=1
)

# Residuals: dots; Residuals are shown as scatter dots — not a line — because they're random noise with no meaningful connection between points
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['resid'],
        mode="markers",
        name="Residuals",
        marker=dict(color="#580A0A", size=8)
    ),
    row=6,
    col=1
)



fig.update_layout(
    height=1000,
    title="Seasonal Decomposition of Professional occupations in engineering [213] in British Columbia - Non-Tuned",
    showlegend=False
)
fig.update_layout(template='ggplot2')
fig.show()

Generating Decompositions

In [ ]:
# Takes a key like 10,1 (province, NOC group), filters the polars dataframe down to just that combination 
# converts to pandas, parses the dates, sorts
# returns a clean 2-column table for that series 
def Get_Sample(Key: tuple):
    try:

        Sample = df.filter(pl.col('Key')==Key).select(['datestamp','employment']).to_pandas()
        Sample['datestamp'] = pd.to_datetime(Sample['datestamp'])
        Sample = Sample.sort_values(by='datestamp', ascending=True)

        return Sample
    
    except Exception as e:
        print(f"Get_Sample(Key={Key}) => {e}")
        return pd.DataFrame()

# real world data can have missing months; MSTL needs a complete unbroken series to work 

# it fills gaps using 3 stratergies: 
# linear interpolation : if a value is missing between two known values, estimate it by drawing a straight line between them 
# forward fill: if there are gaps at the very start, copy the first known value backward
# backward fill:  if there are gaps at the very end, copy the last known value forward
def Process_Sample(Sample: pd.DataFrame):
    try:

        #Interpolate linearly to fill nulls
        Sample['employment'] = Sample['employment'].interpolate(method='linear')

        #Forward fill remaining NaNs at the start of the series
        Sample['employment'] = Sample['employment'].fillna(method='ffill')

        #Backward fill remaining NaNs at the end of the series
        Sample['employment'] = Sample['employment'].fillna(method='bfill')

        return Sample

    except Exception as e:
        print(f"Process_Sample() => {e}")
        return pd.DataFrame()

# sets the date column as the index so MSTL receives a proper time series object
def Convert_Sample(Sample: pd.DataFrame):
    try:

        SampleTS = Sample.sort_values('datestamp')['employment']
        SampleTS.index = Sample['datestamp']
        return SampleTS
    

    except Exception as e:
        print(f"Convert_Sample() => {e}")
        return pd.DataFrame()

# Just wraps the MSTL(periods=[12, 6, 3]).fit() call in a safe try/except
# Returns the result, or None if it fails
def Run_Decomposition(SampleTS: pd.Series):
    try:
        mstl = MSTL(SampleTS, periods=[12, 6, 3])
        result = mstl.fit()

        return result

    except Exception as e:
        print(f"Run_Decompostion => {e}")
        return None
# extracts observed, trend, seasonalities, residual and joins them into one clean DataFrame
# Adds the ds column and resets the index
# Returns the finished decomposition table for one series

def Consolidate_Results(result):
    try:
        observed = result.observed
        trend = result.trend
        seasonalities = result.seasonal
        resid = result.resid

        df_decomp = (
            observed.rename("observed")
            .to_frame()
            .join(trend.rename("trend"))
            .join(resid.rename("resid"))
            .join(seasonalities)  
        )

        

        df_decomp['ds'] = df_decomp.index.values
        df_decomp = df_decomp.reset_index(drop=True)


        return df_decomp

    except Exception as e:
        print(f"Consolidate_Results() => {e}")
        return pd.DataFrame()
# Starts looping over every key with a tqdm progress bar

def main(Keys):
    try:
        All_Decompostions = []
        for key in tqdm(Keys, desc='Processing Series', unit='Keys'):
            provinceid, noc_groupingid = key

            Sample = Get_Sample(Key=key) # fetch the raw series ; skip if empty


            if len(Sample) == 0:
                print(f"Skipping {key}")
                continue

            Sample = Process_Sample(Sample=Sample) # fill any missing values 


            if len(Sample) == 0:
                print(f"Skipping {key}")
                continue

            SampleTS = Convert_Sample(Sample=Sample) #format as time series 


            if len(SampleTS) == 0:
                print(f"Skipping {key}")
                continue

            result = Run_Decomposition(SampleTS=SampleTS) # run MSTL 


            try:
                df_decomp = Consolidate_Results(result=result) # package the output
                df_decomp['provinceid'] = provinceid
                df_decomp['noc_groupingid'] = noc_groupingid

                
                All_Decompostions.append(df_decomp)
            except Exception as e:
                print(f"Skipping {key} => {e}")
            
            
        
        df_All_Decompositions = pd.concat(All_Decompostions, axis=0)
        return df_All_Decompositions

    except Exception as e:
        return pd.DataFrame()

In [109]:
warnings.filterwarnings('ignore')
df_All_Decompositions = main(Keys=Keys)

Processing Series: 100%|██████████| 590/590 [00:14<00:00, 42.02Keys/s]


In [110]:
df_All_Decompositions

,observed,trend,resid,seasonal_3,seasonal_6,seasonal_12,ds,provinceid,noc_groupingid
0,7.0,7.378831,-0.433604,-0.109472,-0.171635,0.335879,2000-01-01,10.0,1
1,8.2,7.410238,0.542999,-0.063244,0.075139,0.234868,2000-02-01,10.0,1
2,7.1,7.440902,0.183164,0.135241,-0.264254,-0.395053,2000-03-01,10.0,1
3,7.6,7.470413,0.108099,-0.047459,0.179786,-0.110839,2000-04-01,10.0,1
4,6.9,7.497976,-0.678920,-0.077763,-0.156244,0.314951,2000-05-01,10.0,1
...,...,...,...,...,...,...,...,...,...
304,15.9,16.568508,-0.454525,-0.237140,-0.991029,1.014186,2025-05-01,59.0,60
305,19.8,17.148172,1.821578,0.557542,-0.774380,1.047088,2025-06-01,59.0,60
306,17.5,17.745231,0.064598,-0.269077,-0.662542,0.621789,2025-07-01,59.0,60
307,18.4,18.357159,-0.750631,-0.561815,1.249591,0.105696,2025-08-01,59.0,60


In [111]:
df_All_Decompositions.to_parquet('Data/SeasonalDecomposedData.parquet',index=False)